<div align="center">
  <a href="https://pageindex.ai/developer">
    <img src="https://github.com/user-attachments/assets/bae02956-6c4e-4a0b-adea-257b0be4aaa1" width="100%" alt="PageIndex — Vectorless, Reasoning-based RAG">
  </a>
  <p>
    <a href="https://pageindex.ai/developer">🌐 Website</a> &nbsp; · &nbsp;
    <a href="https://developer.pageindex.ai/">☁️ Cloud</a> &nbsp; · &nbsp;
    <a href="https://docs.pageindex.ai/">📖 Docs</a> &nbsp; · &nbsp;
    <a href="https://github.com/VectifyAI/PageIndex">⭐ GitHub</a> &nbsp; · &nbsp;
    <a href="https://pageindex.ai/contact">✉️ Contact</a>
  </p>
</div>

---

# PageIndex Indexing Cost Demo

**Index the same document with PageIndex and with a vector index, and compare the cost.**

PageIndex builds a tree index with **gpt-6-luna**. The vector index embeds text chunks with **Gemini Embedding 2**.

1. **Set up**  Install the PageIndex SDK.

2. **Get a document**  Download a 22-page earnings report.

3. **Build the PageIndex tree**  Index the document and calculate the cost.

4. **Build a vector index**  Embed the same text and calculate the cost.

5. **Compare the costs**  Show both side by side.

**Before you start:** You need an OpenAI API key. A Gemini API key is only needed for steps 4 and 5; without one, steps 1–3 still show the PageIndex cost.


## 01 · Set up

Install the latest PageIndex SDK.

In [ ]:
!pip install -U pageindex

## 02 · Get a document

Download the Q1 FY25 earnings report from the PageIndex repository, or set `file_name` to your own PDF.

In [ ]:
import requests
import pypdfium2 as pdfium

url = "https://raw.githubusercontent.com/VectifyAI/PageIndex/main/examples/documents/q1-fy25-earnings.pdf"
file_name = "q1-fy25-earnings.pdf"

response = requests.get(url, timeout=60)
response.raise_for_status()
with open(file_name, "wb") as f:
    f.write(response.content)

pdf = pdfium.PdfDocument(file_name)
pages = [pdf[i].get_textpage().get_text_range() for i in range(len(pdf))]
print(f"{file_name}: {len(pages)} pages")

q1-fy25-earnings.pdf: 22 pages


## 03 · Build the PageIndex tree

Set your OpenAI API key. PageIndex calls its model through LiteLLM, so wrapping LiteLLM's two completion functions counts the tokens of every call.

In [ ]:
import os
import litellm

os.environ["OPENAI_API_KEY"] = "Your OpenAI API key here"

usage = {"input": 0, "output": 0}

def count(response):
    usage["input"] += response.usage.prompt_tokens
    usage["output"] += response.usage.completion_tokens
    return response

acompletion, completion = litellm.acompletion, litellm.completion

async def counted_acompletion(**kwargs):
    return count(await acompletion(**kwargs))

litellm.acompletion = counted_acompletion
litellm.completion = lambda **kwargs: count(completion(**kwargs))

Index the document, then price the calls. Every input token is priced at the standard rate, as for a document indexed for the first time; re-indexing the same PDF can hit OpenAI's prompt cache, which bills part of the input at a tenth of the price.

In [ ]:
from pageindex import PageIndexClient

# gpt-6-luna, standard USD per 1M tokens. Checked 2026-10-12:
# https://developers.openai.com/api/docs/models/gpt-6-luna
LUNA_INPUT, LUNA_OUTPUT = 0.10, 0.50

client = PageIndexClient(index="gpt-6-luna")
doc_id = client.submit_document(file_name)["doc_id"]

pageindex_cost = (usage["input"] * LUNA_INPUT + usage["output"] * LUNA_OUTPUT) / 1_000_000
print(f"Indexed {doc_id}: {usage['input']:,} input and {usage['output']:,} output tokens")
print(f"PageIndex: ${pageindex_cost:.4f} for this PDF, "
      f"${pageindex_cost / len(pages) * 1000:.3f} per 1,000 pages")

Indexed pi-0644c6a6fe534cfba7414db5a0f3a9f5: 16,808 input and 3,691 output tokens
PageIndex: $0.0035 for this PDF, $0.160 per 1,000 pages


## 04 · Build a vector index

Set your Gemini API key. Split the same text into 512-token chunks with a 50-token overlap, a common vector-RAG setting, and embed them with Gemini Embedding 2. The REST response reports the tokens Google bills.

In [ ]:
import os
import requests
import tiktoken

os.environ["GEMINI_API_KEY"] = "Your Gemini API key here"

# Gemini Embedding 2, standard USD per 1M tokens. Checked 2026-10-12:
# https://ai.google.dev/gemini-api/docs/pricing
GEMINI_EMBEDDING = 0.20

enc = tiktoken.get_encoding("cl100k_base")
CHUNK, OVERLAP = 512, 50
ids = enc.encode("".join(pages), disallowed_special=())
chunks = [enc.decode(ids[i:i + CHUNK]) for i in range(0, len(ids), CHUNK - OVERLAP)]

embedding_tokens = 0
for i in range(0, len(chunks), 100):
    response = requests.post(
        "https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-2:batchEmbedContents",
        headers={"x-goog-api-key": os.environ["GEMINI_API_KEY"]},
        json={"requests": [{"model": "models/gemini-embedding-2",
                            "content": {"parts": [{"text": chunk}]}}
                           for chunk in chunks[i:i + 100]]},
        timeout=120)
    response.raise_for_status()
    embedding_tokens += response.json()["usageMetadata"]["promptTokenCount"]

vector_cost = embedding_tokens * GEMINI_EMBEDDING / 1_000_000
print(f"{len(chunks)} chunks, {embedding_tokens:,} billed tokens")
print(f"Vector index: ${vector_cost:.4f} for this PDF, "
      f"${vector_cost / len(pages) * 1000:.3f} per 1,000 pages")

32 chunks, 20,731 billed tokens
Vector index: $0.0041 for this PDF, $0.188 per 1,000 pages


## 05 · Compare the costs

Put steps 03 and 04 side by side.

In [ ]:
print(f"{'':28}{'this PDF':>12}{'per 1,000 pages':>18}")
for name, cost in (("PageIndex (gpt-6-luna)", pageindex_cost),
                   ("Vector index (Gemini Emb. 2)", vector_cost)):
    print(f"{name:28}{f'${cost:.4f}':>12}{f'${cost / len(pages) * 1000:.3f}':>18}")
print(f"\nPageIndex costs {pageindex_cost / vector_cost:.2f}x the vector index to build.")

                                this PDF   per 1,000 pages
PageIndex (gpt-6-luna)           $0.0035            $0.160
Vector index (Gemini Emb. 2)     $0.0041            $0.188

PageIndex costs 0.85x the vector index to build.


The vector side counts the embedding calls only; storing and serving the vectors in a vector database is extra.

---

<div align="center">
  <br>
  <p>
    <a href="https://docs.pageindex.ai/"><b>Upload your documents</b></a>
    &nbsp;&nbsp; · &nbsp;&nbsp;
    <a href="https://docs.pageindex.ai/sdk/agents"><b>Integrate with your agent</b></a>
    &nbsp;&nbsp; · &nbsp;&nbsp;
    <a href="https://github.com/VectifyAI/PageIndex"><b>Star PageIndex on GitHub</b></a>
  </p>
  <br>
  <a href="https://pageindex.ai">
    <img src="https://github.com/user-attachments/assets/e1c677f2-b590-4ffc-859d-e8cc7d794bdf" width="144" height="40" alt="PageIndex">
  </a>
</div>
